In [104]:
import json
from pathlib import Path

from collections import defaultdict

from app.schemas import ModelEvaluation, GradingResult, KeyConcepts

In [105]:
dir_path = Path("../output/full_grading")

In [106]:
data: list[ModelEvaluation] = []

for model_dir in dir_path.iterdir():
    if model_dir.is_file():
        continue

    for evaluation_file in model_dir.iterdir():
        if evaluation_file.is_dir():
            continue

        content = evaluation_file.read_text(encoding="utf8")
        model_evaluation = ModelEvaluation.model_validate(json.loads(content))
        data.append(model_evaluation)

In [107]:
def calc_score(grading_result: GradingResult, key_concepts: KeyConcepts):
    global wrong
    global correct
    score = 0
    for i, concept_score in enumerate(grading_result.concepts):
        try:
            concept = key_concepts.concepts[concept_score.id - 1]
        except IndexError as e:
            raise ValueError("Wrong concept id")
        
        score += concept_score.coverage * concept.importance

    total_possible = sum(x.importance for x in key_concepts.concepts) * 2

    return score / total_possible * 100

In [108]:
metrics = defaultdict(lambda: defaultdict(float))

In [109]:
import pandas as pd

In [110]:
tolerance = 10

In [111]:
scores = pd.DataFrame(columns=["model", "answer_type", "real_score", "model_score"])

for item in data:
    try:
        grade_score = calc_score(item.grading_results, item.key_concepts)
    except ValueError as e:
        continue

    real_score = item.answer.quality

    error = real_score - grade_score
    metrics[item.model]["mae"] += abs(error)
    metrics[item.model]["mse"] += error**2
    metrics[item.model]["valid_examples"] += 1

    if real_score - tolerance <= grade_score <= real_score + tolerance:
        metrics[item.model]["correct estimation"] += 1
    elif grade_score < real_score - tolerance * 2:
        metrics[item.model]["super low estimation"] += 1
    elif grade_score > real_score + tolerance * 2:
        metrics[item.model]["super over estimation"] += 1
    elif grade_score < real_score - tolerance:
        metrics[item.model]["low estimation"] += 1
    else:
        metrics[item.model]["over estimation"] += 1

    new_row = pd.DataFrame([{"model": item.model, "answer_type": item.answer.answer_type, "real_score": real_score, "model_score": grade_score}])

    scores = pd.concat([scores, new_row], ignore_index=True)

In [112]:
for model in metrics:
    metrics[model]["mae"] /= metrics[model]["valid_examples"]
    metrics[model]["mse"] /= metrics[model]["valid_examples"]

In [113]:
metrics

defaultdict(<function __main__.<lambda>()>,
            {'llama3-8b': defaultdict(float,
                         {'mae': 17.401798822231502,
                          'mse': 504.18948647043123,
                          'valid_examples': 499.0,
                          'correct estimation': 207.0,
                          'super over estimation': 53.0,
                          'low estimation': 81.0,
                          'over estimation': 42.0,
                          'super low estimation': 116.0}),
             'tlite-8b': defaultdict(float,
                         {'mae': 18.72331945821871,
                          'mse': 600.0968255032813,
                          'valid_examples': 500.0,
                          'low estimation': 93.0,
                          'correct estimation': 204.0,
                          'super low estimation': 156.0,
                          'over estimation': 25.0,
                          'super over estimation': 22.0}),
           

In [115]:
for model in scores["model"].unique():
    print(model)
    corr = scores[scores.model == model]["real_score"].corr(scores[scores.model == model]["model_score"], method="spearman")
    print(corr)

llama3-8b
0.7611439933944707
tlite-8b
0.8221905734557207
vikhr-8b
0.7377093685444558
